# Generate an Evaluation Dataset for a RAG Agent

In this notebook, you'll create a synthetic evaluation dataset using **NVIDIA NeMo Data Designer**, an open source tool for synthetic data generation. We'll generate test cases for evaluating the **IT Help Desk RAG Agent** from Module 2.

## 1. Setup

First, let's install the Data Designer library and set up our environment.

In [ ]:
# Install Data Designer if needed
%pip install data-designer -q

# Load environment variables
from dotenv import load_dotenv
_ = load_dotenv("../../variables.env")
_ = load_dotenv("../../secrets.env")

import os
import json
from pathlib import Path

print("Completed setup")

Note: you may need to restart the kernel to use updated packages.
Completed setup


## 2. Configure DataDesigner

Next, initialize Data Designer with default settings.


In [2]:
from data_designer.essentials import (
    DataDesigner,
    DataDesignerConfigBuilder,
    LLMTextColumnConfig,
)

# Initialize Data Designer with default settings
data_designer = DataDesigner()
config_builder = DataDesignerConfigBuilder()

print("Completed initialization")


/home/nvidia/workshop-build-an-agent/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Completed initialization


## 3. Load IT Knowledge Base as Seed Data

We’ll use articles from the IT knowledge base as seed data for Data Designer. Seed data is the initial set of examples Data Designer uses to guide what it generates.

By seeding Data Designer with our agent's knowledge base, we'll ensure the evaluation data is grounded in information the agent is expected to know.


In [3]:
import pandas as pd

# Load all articles from the knowledge base directory
kb_dir = Path("../../data/it-knowledge-base")
kb_articles = []

for kb_file in sorted(kb_dir.glob("*.md")):
    with open(kb_file, 'r') as f:
        content = f.read()
    
    # Extract the article title and create a readable topic name
    topic_name = kb_file.stem.replace('-', '_')
    title = content.split('\n')[0].replace('# ', '').strip()
    
    kb_articles.append({
        "topic": topic_name,
        "title": title,
        "filename": kb_file.name,
        "content": content
    })

# Create seed data DataFrame
seed_data = pd.DataFrame(kb_articles)

# Data Designer requires seed data to be saved as a file for efficient sampling
seed_file_path = Path("../../data/evaluation/kb_seed_data.parquet")
seed_ref = data_designer.make_seed_reference_from_dataframe(
    dataframe=seed_data,
    file_path=seed_file_path
)

print(f"Created seed reference with {len(seed_data)} KB articles")

[00:13:33] [INFO] 💾 Saving seed dataset to ../../data/evaluation/kb_seed_data.parquet


Created seed reference with 12 KB articles


## 4. Configure Dataset Structure for RAG Evaluation

We'll use Data Designer’s Config Builder to define the structure of our synthetic dataset. Our configuration will include the following components:

- Category (sampled from a list of knowledge base topics)
- IT help desk questions (based on category)
- Context keywords (to evaluate retrieval)
- Ground truth answers (ideal agent answer)

We do this by defining a list of "columns" representing the different fields of data that we want to generate. Using Jinja formatting, we can reference items in other columns in the same "row." This allows us to, for example, generate a question and then generate an answer based on that question.

In [4]:
# Create config builder with seed data
config_builder = DataDesignerConfigBuilder()
config_builder.with_seed_dataset(seed_ref)

# Generate question based on the seed article
config_builder.add_column(
    LLMTextColumnConfig(
        name="question",
        model_alias="nvidia-text",
        prompt="""Generate a realistic IT help desk question that a user might ask about the topic {{ topic }} from article {{ title }}.

The question should:
- Sound like a real user would ask it
- Be specific and clear
- Cover common scenarios related to this topic
- Be directly answerable using the information in this KB article
- NOT just restate the article title

Return ONLY the question, nothing else.""",
    )
)

# Generate context keywords for retrieval verification
config_builder.add_column(
    LLMTextColumnConfig(
        name="context_key_words",
        model_alias="nvidia-text",
        prompt="""Based on this IT help desk question: {{ question }}

List 2-3 key words or phrases that would help locate the relevant knowledge base article.

Return as a comma-separated list.
Return ONLY the keywords, nothing else.""",
    )
)

# Generate ground truth answer based on the knowledge base
config_builder.add_column(
    LLMTextColumnConfig(
        name="ground_truth",
        model_alias="nvidia-text",
        prompt="""Based on this IT help desk question: {{ question }}

And this knowledge base article content:
{{ content }}

Generate a concise, accurate answer that directly addresses the question using only information from the knowledge base.

Return ONLY the answer, nothing else.""",
    )
)

print("Dataset configuration built")

Dataset configuration built


## 5. Preview the Data Structure

Let's preview what the generated test cases will look like.

In [5]:
# Preview a few sample records to see what will be generated
preview = data_designer.preview(config_builder=config_builder, num_records=3)
preview.display_sample_record()


[00:13:33] [INFO] 👀 Preview generation in progress
[00:13:33] [INFO] ✅ Validation passed
[00:13:33] [INFO] ⛓️ Sorting column configs into a Directed Acyclic Graph
[00:13:33] [INFO] 🩺 Running health checks for models...
[00:13:33] [INFO]   |-- 👀 Checking 'nvidia/nvidia-nemotron-nano-9b-v2' in provider named 'nvidia' for model alias 'nvidia-text'...
[00:13:34] [INFO]   |-- ✅ Passed!
[00:13:34] [INFO] 🌱 Sampling 3 records from seed dataset
[00:13:34] [INFO]   |-- seed dataset size: 12 records
[00:13:34] [INFO]   |-- sampling strategy: ordered
[00:13:34] [INFO] 📝 Preparing llm-text column generation
[00:13:34] [INFO]   |-- column name: 'question'
[00:13:34] [INFO]   |-- model config:
{
    "alias": "nvidia-text",
    "model": "nvidia/nvidia-nemotron-nano-9b-v2",
    "inference_parameters": {
        "temperature": 0.85,
        "top_p": 0.95,
        "max_tokens": null,
        "max_parallel_requests": 4,
        "timeout": null,
        "extra_body": null
    },
    "provider": "nvidia"
}

                                                   Seed Columns                                                    
┏━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Name     ┃ Value                                                                                                ┃
┡━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ topic    │ email_distribution_lists                                                                             │
├──────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────┤
│ title    │ Managing Email Distribution Lists                                                                    │
├──────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────┤
│ filename │ email-distribution-lists.md                                                                          │
├──────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────┤
│ content  │ # Managing Email Distribution Lists                                                                  │
│          │                                                                                                      │
│          │ ## Purpose                                                                                           │
│          │                                                                                                      │
│          │ This article provides instructions for requesting, managing, and using email distribution lists      │
│          │ within the company's email system. Distribution lists enable efficient communication with groups of  │
│          │ employees, departments, or project teams while maintaining proper security and governance controls.  │
│          │                                                                                                      │
│          │ ## What are Email Distribution Lists                                                                 │
│          │                                                                                                      │
│          │ Email distribution lists are centrally managed groups of email addresses that allow you to send      │
│          │ messages to multiple recipients using a single email address. Distribution lists provide:            │
│          │                                                                                                      │
│          │ - **Simplified group communication** for departments, projects, and teams                            │
│          │ - **Centralized membership management** with proper access controls                                  │
│          │ - **Consistent naming conventions** for easy identification                                          │
│          │ - **Security and compliance** with company communication policies                                    │
│          │ - **Automatic synchronization** with organizational changes                                          │
│          │                                                                                                      │
│          │ ## Types of Distribution Lists                                                                       │
│          │                                                                                                      │
│          │ ### **Departmental Lists**                                                                           │
│          │ - **Purpose:** Communication within specific departments or business units                           │
│          │ - **Naming Convention:** `dept-[department-name]@company.com`                                        │
│          │ - **Examples:** `dept-finance@company.com`,

## 6. Generate the Full Dataset

Now let's generate the complete evaluation dataset.

In [6]:
# We'll only generate a few records for this tutorial
dataset_results = data_designer.create(
    config_builder=config_builder,
    num_records = 10,
    dataset_name="rag_agent_test_cases"
)

# Convert to list of dictionaries for processing
eval_data = dataset_results.load_dataset().to_dict('records')

print(f"Generated {len(eval_data)} test cases")


[00:13:52] [INFO] 🎨 Creating Data Designer dataset
[00:13:52] [INFO] ✅ Validation passed
[00:13:52] [INFO] ⛓️ Sorting column configs into a Directed Acyclic Graph
[00:13:52] [INFO] 📂 Dataset path '/home/nvidia/workshop-build-an-agent/code/3-agent-evaluation/artifacts/rag_agent_test_cases' already exists. Dataset from this session
		     will be saved to '/home/nvidia/workshop-build-an-agent/code/3-agent-evaluation/artifacts/rag_agent_test_cases_12-11-2025_001352' instead.
[00:13:52] [INFO] 🩺 Running health checks for models...
[00:13:52] [INFO]   |-- 👀 Checking 'nvidia/nvidia-nemotron-nano-9b-v2' in provider named 'nvidia' for model alias 'nvidia-text'...
[00:13:53] [INFO]   |-- ✅ Passed!
[00:13:53] [INFO] ⏳ Processing batch 1 of 1
[00:13:53] [INFO] 🌱 Sampling 10 records from seed dataset
[00:13:53] [INFO]   |-- seed dataset size: 12 records
[00:13:53] [INFO]   |-- sampling strategy: ordered
[00:13:53] [INFO] 📝 Preparing llm-text column generation
[00:13:53] [INFO]   |-- column name: '

Generated 10 test cases


## 7. Save the Test Data

Now let's clean up and save the generated data in a format ready for evaluation.

We'll keep only the essential evaluation fields: **question**, **context_key_words**, and **ground_truth**. We'll remove the seed data and reasoning traces to make the output cleaner and easier to review.

In [7]:
# Set output path
output_path = Path("../../data/evaluation/synthetic_rag_agent_test_cases.json")

# Keep only essential evaluation columns, ignore reasoning traces and seed data
essential_columns = ['question', 'context_key_words', 'ground_truth']
clean_data = []

for record in eval_data:
    clean_record = {}
    for key, value in record.items():
        if key in essential_columns:
            clean_record[key] = value
    clean_data.append(clean_record)

# Save to JSON file
with open(output_path, 'w') as f:
    json.dump(clean_data, f, indent=2)

print(f"Saved {len(eval_data)} test cases to {output_path}")

Saved 10 test cases to ../../data/evaluation/synthetic_rag_agent_test_cases.json


## What's Next?

Check out <button onclick="openOrCreateFileInJupyterLab('data/evaluation/synthetic_rag_agent_test_cases.json');"><i class="fa-brands fa-python"></i> synthetic_rag_agent_test_cases.json </button> to view your generated data.

In the next notebook, you'll use this same process to create an evaluation dataset for your Report Generation agent!